# Homework 5: Foundation Models, Data Preparation

This unit builds a small **video-understanding pipeline** on a German
documentary clip from the [Planet Wissen](https://commons.wikimedia.org/wiki/Category:Planet_Wissen)
series on Wikimedia Commons, using three open models:

| Stage | Model | Notebook |
|-------|-------|----------|
| Speech → text | [`openai/whisper-large-v3`](https://huggingface.co/openai/whisper-large-v3) | this one (`00`) |
| German → English translation | [`Qwen/Qwen2.5-VL-7B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) | `01_Translate` |
| Summary + on-screen-text OCR | [`Qwen/Qwen2.5-VL-7B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) | `02_Summarize_OCR` |

In this first notebook we **download** a Planet Wissen clip **of your choice**, **convert** it to MP4, and
**transcribe** the German speech with Whisper Large-V3.

**Sources**
- Video: [Video of your choice from wikimedia](https://commons.wikimedia.org/wiki/Category:Planet_Wissen) — Planet Wissen, Wikimedia Commons
- Whisper Large-V3: <https://huggingface.co/openai/whisper-large-v3> ([paper](https://arxiv.org/abs/2212.04356))
- Qwen2.5-VL: <https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct> ([report](https://arxiv.org/abs/2502.13923))
- FFmpeg: <https://ffmpeg.org/> · `qwen-vl-utils`: <https://pypi.org/project/qwen-vl-utils/>

## Prerequisites

**1. FFmpeg** (used to convert the video and to extract audio/frames). It is a
system package, *not* a Python one, so install it with `apt`:

```bash
sudo apt update
sudo apt install -y ffmpeg
ffmpeg -version      # verify it is on your PATH
```

**2. The Python environment** for this unit (managed with [uv](https://docs.astral.sh/uv/)).
From this directory:

```bash
uv sync                 # create .venv with torch (CUDA 12.8), transformers, etc.
uv run jupyter lab      # launch notebooks inside the environment
```

> **GPU:** Whisper Large-V3 (~3 GB) and Qwen2.5-VL-7B (~16 GB in fp16) are large.
> A CUDA GPU is strongly recommended; the models are loaded one at a time and
> unloaded afterwards so they never share GPU memory.

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

# Make the vendored `video_pipeline` package importable even if the project
# was not installed (e.g. running `jupyter` outside `uv run`).
SRC = Path.cwd() / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PREPARED = Path("prepared")
PREPARED.mkdir(exist_ok=True)

if shutil.which("ffmpeg") is None:
    raise SystemExit(
        "ffmpeg not found on PATH. Install it first (see the cell above):\n"
        "    sudo apt update && sudo apt install -y ffmpeg"
    )
print("ffmpeg:", shutil.which("ffmpeg"))

ffmpeg: /anaconda/envs/azureml_py38/bin/ffmpeg


## 1. Download the video

We pull the `.webm` straight from Wikimedia Commons with `wget`. The
`Special:FilePath` URL redirects to the current media file, so we don't have to
hard-code the storage path.

In [2]:
# TODO (Exercise 1a): pick ANY clip from the Planet Wissen category on Commons
#   https://commons.wikimedia.org/wiki/Category:Planet_Wissen
# and fill in its exact file name (the part after "File:").
COMMONS_TITLE = "Die_Erfindungen_von_Leonardo_da_Vinci_-_Planet_Wissen.webm" # e.g. "Der_Golfstrom_-_Planet_Wissen.webm"
STEM = "Leonardo_da_Vinci" # short local name, e.g. "der_golfstrom"

webm_path = PREPARED / f"{STEM}.webm"
url = f"https://commons.wikimedia.org/wiki/Special:FilePath/{COMMONS_TITLE}"
if not webm_path.exists():
    subprocess.run(["wget", "-nv", "-c", "-O", str(webm_path), url], check=True)
print(webm_path, webm_path.stat().st_size // 1024, "KiB")

prepared/Leonardo_da_Vinci.webm 155640 KiB


2026-06-27 09:19:47 URL:https://upload.wikimedia.org/wikipedia/commons/0/01/Die_Erfindungen_von_Leonardo_da_Vinci_-_Planet_Wissen.webm [159376176/159376176] -> "prepared/Leonardo_da_Vinci.webm" [1]


## 2. Convert WebM → MP4

The pipeline works with `.mp4`/`.mkv` containers. We re-encode the VP8/Vorbis
WebM to **H.264 video + AAC audio**, which is the most widely compatible MP4
profile. This is the FFmpeg command from the project brief:

```bash
ffmpeg -i input.webm -c:v libx264 -c:a aac -strict -2 output.mp4
```

In [5]:
# TODO (Exercise 1b): complete the ffmpeg command to transcode to H.264 + AAC.
mp4_path = PREPARED / f"{STEM}.mp4"
if not mp4_path.exists():
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error", "-nostdin", "-i", str(webm_path),
        "-c:v", "libopenh264", "-c:a", "aac", "-strict", "-2",
        str(mp4_path),
    ]
    subprocess.run(cmd, check=True)
print(mp4_path, mp4_path.stat().st_size // 1024, "KiB")

prepared/Leonardo_da_Vinci.mp4 26151 KiB


In [6]:
from video_pipeline.audio_extractor import probe_duration

duration = probe_duration(mp4_path)
print(f"duration: {duration:.1f} s")

duration: 100.1 s


## 3. Transcribe the German speech (Whisper Large-V3)

`WhisperPipeline` extracts a 16 kHz mono WAV with FFmpeg and runs Whisper
Large-V3, returning timestamped segments. We force `language="german"` so the
detector can't drift. The model is unloaded from the GPU on `with`-block exit.

> Equivalent one-liner on the command line:
> ```bash
> uv run video-pipeline whisper prepared/STEM.mp4 -l german -o prepared/transcript.json
> ```

In [7]:
from video_pipeline.whisper_pipeline import WhisperPipeline

transcript_path = PREPARED / "transcript.json"
# TODO (Exercise 1c): transcribe `mp4_path`. The Planet Wissen clips are German,
# so force the language. Look at WhisperPipeline.transcribe(...).
with WhisperPipeline() as wp:
    transcript = wp.transcribe(mp4_path, language="german")

transcript_path.write_text(transcript.model_dump_json(indent=2), encoding="utf-8")
print(f"{len(transcript.segments)} segments, {len(transcript.text)} chars -> {transcript_path}")

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[trans

9 segments, 1698 chars -> prepared/transcript.json


In [8]:
# Peek at the first few timestamped segments.
for seg in transcript.segments[:6]:
    print(f"[{seg.start:6.1f} - {seg.end:6.1f}s]  {seg.text}")

[   1.0 -   27.3s]  Das italienische Universal-Genie Leonardo da Vinci hat nicht nur die Mona Lisa und andere schöne Bilder gemalt, er war auch ein genialer Ingenieur. Viele Skizzen und Beschreibungen seiner Erfindungen sind bis heute erhalten. Zahnräder spielten bei seinen Erfindungen eine große Rolle. Das Zahnrad an sich existierte bereits, aber da Vinci entwickelte daraus zahlreiche Maschinen. Zum Beispiel das Kettenradgetriebe, das wir heute in jedem Fahrrad finden.
[  28.2 -   33.2s]  Oder einen Schwerlastheber. Etwas abgewandelt entspricht das dem heutigen Wagenheber.
[  34.8 -   37.5s]  Viele Waffen entwickelte er im Auftrag seiner Arbeitgeber.
[  38.0 -   42.1s]  Zum Beispiel dieses Orgelgeschütz. Quasi das erste Maschinengewehr.
[  43.5 -   48.9s]  Und sogar einen Panzer, der sich allerdings wegen eines Konstruktionsfehlers nicht vorwärts bewegen konnte.
[  49.5 -   48.9s]  


---
**Next:** `01_Translate.ipynb` — translate this German transcript into English
with Qwen2.5-VL.